In [ ]:
from collections import Counter
from pathlib import Path
import re
import matplotlib.pyplot as plt
import pandas as pd
from wordcloud import STOPWORDS, WordCloud

# 1. Carregamento dos dados
caminho = (
    Path("artigos.parquet")
    if Path("artigos.parquet").exists()
    else Path("../data/exemplo/artigos.parquet")
)
df = pd.read_parquet(caminho)

col_texto = next(
    (c for c in ["resumo", "titulo_artigo", "titulo"] if c in df.columns),
    df.select_dtypes(include="object").columns[0],
)

# --- AUDITORIA: ESTADO BRUTO (ANTES DAS MUDANÇAS) ---
print("=" * 60)
print(f"1. ESTADO BRUTO DOS DADOS TEXTUAIS (Coluna: '{col_texto}')")
print("=" * 60)
texto_bruto_amostra = df[col_texto].dropna().astype(str)
print(f"Total de registros analisados: {len(texto_bruto_amostra)}")
print("\nAmostra dos textos originais sem tratamento:")
for i, t in enumerate(texto_bruto_amostra.head(3), 1):
    print(f"  [{i}] {t}")

# 2. TRATAMENTO E LIMPEZA DE TEXTO
texto_unificado = " ".join(texto_bruto_amostra).lower()
texto_limpo = re.sub(r"[^\w\s]", "", texto_unificado)

# Configuração de stopwords (palavras irrelevantes em português)
stopwords_pt = set(STOPWORDS)
stopwords_pt.update(
    [
        "de",
        "da",
        "do",
        "em",
        "para",
        "com",
        "e",
        "o",
        "a",
        "os",
        "as",
        "no",
        "na",
        "nos",
        "nas",
        "um",
        "uma",
        "dos",
        "das",
        "por",
        "se",
        "que",
        "como",
        "sobre",
        "ao",
        "aos",
        "estudo",
        "analise",
        "uso",
    ]
)

palavras_brutas = texto_limpo.split()
palavras_filtradas = [p for p in palavras_brutas if p not in stopwords_pt]

# --- AUDITORIA: MUDANÇAS APLICADAS (DEPOIS DA LIMPEZA) ---
print("\n" + "=" * 60)
print("2. MUDANÇAS APLICADAS E RELATÓRIO DE DEPURACÃO")
print("=" * 60)
print(f"✓ Palavras totais extraídas (bruto): {len(palavras_brutas)}")
print(f"✓ Palavras mantidas (pós-limpeza/stopwords): {len(palavras_filtradas)}")
print(
    f"✓ Termos desconsiderados (ruído removido): {len(palavras_brutas) - len(palavras_filtradas)}"
)

contador = Counter(palavras_filtradas)
print("\nTOP 10 TERMOS MAIS RECORRENTES:")
for termo, freq in contador.most_common(10):
    print(f"  • {termo.upper()}: {freq} ocorrências")
print("=" * 60 + "\n")

# 3. GERAÇÃO DA NUVEM DE PALAVRAS
nuvem = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    colormap="Blues_r",
    max_words=100,
    stopwords=stopwords_pt,
).generate(" ".join(palavras_filtradas))

plt.figure(figsize=(12, 6))
plt.imshow(nuvem, interpolation="bilinear")
plt.axis("off")
plt.title(
    f"Nuvem de Palavras — Termos Mais Frequentes em '{col_texto}'",
    fontsize=14,
    pad=15,
)
plt.tight_layout()
plt.show()

**Análise de Mineração de Texto e Nuvem de Palavras:**
* **Termos Dominantes:** As palavras em maior tamanho na imagem correspondem aos tópicos centrais de pesquisa do conjunto de publicações.
* **Efetividade do Filtro:** A conversão de caixa e o expurgo das *stopwords* em português removeram conectivos e numerais, garantindo que a visualização final contenha apenas termos conceituais relevantes.